# V0.2 Mid-task Crash Recovery Lab

问题：Agent 跑一半程序崩了，为什么还能继续？

这个实验故意在任务未完成时崩溃：ToolResult 已经写入 durable Session，但最终 Assistant answer 还没有写入。

In [ ]:
MODE = "deterministic"

from pathlib import Path
import sys

def find_agentkernel_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir() and (path / "labs").is_dir():
            return path
    raise RuntimeError("Run this notebook from the AgentKernel repo root or the labs directory.")

REPO_ROOT = find_agentkernel_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from labs import create_lab

lab = create_lab("v02", mode=MODE)


## Step 1: Runtime P1 starts

P1 是当前 live runtime/process；Session 是 durable truth。

In [ ]:
lab.setup()

## Step 2: What does P1 send to the model?

In [ ]:
lab.show_model_request()

预测：模型会提出 `math.add` ToolCall，还是直接回答？

In [ ]:
lab.model_step()

## Step 3: ToolResult durable, task unfinished

下一格执行 ToolCall 并写入 ToolResult，但故意不写最终 Assistant response。

In [ ]:
lab.execute_tool_before_crash()

## Step 4: Crash now

现在 live runtime/process P1 会被丢弃。注意：Session 文件仍然存在。

In [ ]:
lab.crash()

## Step 5: New runtime P2 resumes same Session

观察：Process/runtime 换了，但 Session id 没变，并且 ToolResult 被投影回下一次 model-visible request。

In [ ]:
lab.restart()

## Step 6: Continue unfinished task

In [ ]:
lab.continue_after_restart()

## Summary

In [ ]:
lab.summary()